# Survey associations

Run from the repository root after building the database from the prepared data bundle; see README.md.

**Private inputs required:** survey responses and respondent-to-sensor linkage are not distributed. Request access from the authors. Set `DELFT_SURVEY_ROOT` to a directory containing `survey.sav` and `survey-contacts.csv`. Without both files this notebook stops with an explanatory error.


# Initialization and database connection

In [ ]:
from analysis_paths import connect_analysis, data_root, output_path, require_file
import duckdb
import pandas as pd
import numpy as np
from scipy.stats import permutation_test, rankdata, spearmanr
from statsmodels.stats.multitest import multipletests

# Connect to the prepared DuckDB database
db = connect_analysis()


print("DuckDB version:", db.execute("SELECT version()" ).fetchone()[0])
from analysis_paths import survey_input


# Noise survey

## Survey preprocessing

In [ ]:
df_survey = pd.read_spss(survey_input('survey.sav'))

# add sensor ids
contacts = pd.read_csv(survey_input('survey-contacts.csv'))
contacts['sensor_id'] = pd.to_numeric(contacts['SensorID'], errors='coerce').map(lambda v: f"citizen-{int(v):03d}" if pd.notna(v) else np.nan)
df_survey = df_survey.merge(contacts[['ExternalDataReference','sensor_id']], on='ExternalDataReference', how='left')

# encode categorical variables numerically
common_map = {
    'Not at all common': 1,
    'Slightly common': 2,
    'Moderately common': 3,
    'Very common': 4,
    'Extremely common': 5,
}
annoying_map = {
    'Not at all': 1,
    'Slightly': 2,
    'Moderately': 3,
    'Very': 4,
    'Extremely': 5,
}
agree_map = {
    'Strongly disagree': 1,
    'Disagree': 2,
    'Slightly disagree': 3,
    'Slightly agree': 4,
    'Agree': 5,
    'Neutral': 3,
    'Strongly agree': 5,
}
applies_map = {
    'Does not apply': 1,
    'Applies': 5, 
}

prefix_map = {
    'common_sources_': common_map,
    'annoying_sources_': annoying_map,
    'noise_sensitivity_': agree_map,
    'living_env_': applies_map,
    'annoying_features_': annoying_map,
}

# Discover columns by prefix
cols = [c for c in df_survey.columns if any(c.startswith(pref) for pref in prefix_map)]

encoded_columns = []
for col2 in cols:
    # Pick the mapping for this prefix
    mapper = next((mp for pref, mp in prefix_map.items() if col2.startswith(pref)), None)
    if mapper is None:
        continue
    ser = (
        df_survey[col2].map(mapper).astype('Int64')
    )
    df_survey[f'{col2}_num'] = ser
    encoded_columns.append(f'{col2}_num')

# Rename common source labels
prefix = 'common_sources_'
rename_map = {
    prefix + '1_num': prefix + 'traffic',
    prefix + '2_num': prefix + 'aviation',
    prefix + '3_num': prefix + 'construction',
    prefix + '4_num': prefix + 'voices',
    prefix + '5_num': prefix + 'neighbour',
    prefix + '6_num': prefix + 'bird',
    prefix + '7_num': prefix + 'emergency',
    prefix + '8_num': prefix + 'other',
}

df_survey.rename(columns=rename_map, inplace=True)

# Rename annoying source labels
prefix = 'annoying_sources_'
rename_map = {
    prefix + '1_num': prefix + 'traffic',
    prefix + '2_num': prefix + 'aviation',
    prefix + '3_num': prefix + 'construction',
    prefix + '4_num': prefix + 'voices',
    prefix + '5_num': prefix + 'neighbour',
    prefix + '6_num': prefix + 'emergency',
    prefix + '7_num': prefix + 'other',
}

df_survey.rename(columns=rename_map, inplace=True)

# Rename annoying features labels
prefix = 'annoying_features_'
rename_map = {
    prefix + '1_num': prefix + 'volume',
    prefix + '2_num': prefix + 'bursts',
    prefix + '3_num': prefix + 'low_freq',
    prefix + '4_num': prefix + 'high_freq',
    prefix + '5_num': prefix + 'daytime',
    prefix + '6_num': prefix + 'nighttime',
}

df_survey.rename(columns=rename_map, inplace=True)

# Rename living environment labels
prefix = 'living_env_'
rename_map = {
    prefix + '1_num': prefix + 'outside_space',
    prefix + '2_num': prefix + 'bedroom_street_facing',
    prefix + '3_num': prefix + 'living_room_street_facing',
    prefix + '4_num': prefix + 'window_insulation',
}

df_survey.rename(columns=rename_map, inplace=True)

In [ ]:
# Count missing values
missing_counts = df_survey.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
missing_counts.head(20)

In [ ]:
df_survey.columns

In [ ]:
# Noise sensitivity score: sum items 1,3,4,5 + reverse-coded item 2 (6 - x)
cols = ['noise_sensitivity_1_num','noise_sensitivity_3_num','noise_sensitivity_4_num','noise_sensitivity_5_num']
df_survey['noise_sensitivity_score'] = (6 - df_survey['noise_sensitivity_2_num']) + df_survey[cols].sum(axis=1)
df_survey[['sensor_id','noise_sensitivity_score']].head()

df_survey.hist('noise_sensitivity_score')

In [ ]:
# Survey sample size
n_respondents = df_survey['ExternalDataReference'].nunique()
n_unique_sensors = df_survey.loc[df_survey['sensor_id'].notna(), 'sensor_id'].nunique()

print(f"Unique respondents: {n_respondents}")
print(f"Unique sensor-linked respondents: {n_unique_sensors}")

## Spearman correlation of surveyed sources and sensor detections

For intermittent events, the source-specific conditional presence ratio is the number of qualifying intermittent intervals containing a detection of the source class divided by all qualifying intermittent intervals. The existing hourly eligibility rule (at least 300 valid 3-second intervals), threshold (10 dB above the hourly energy-equivalent level), and class-detection threshold (0.5) are retained. A sensor with no qualifying intermittent intervals has an undefined ratio rather than zero.


In [ ]:
def source_survey_spearman(source='vehicle', common_col='common_sources_traffic', prob_thresh=0.5, loudness_min_dba=60, print_output=True):
    df_merged = db.sql(f"""
      SELECT
        sensor_id,
        100.0*AVG(({source} > {prob_thresh})::DOUBLE) AS pct
      FROM sources
      WHERE
        avg_dba > {loudness_min_dba}
      GROUP BY sensor_id
    """).df().merge(df_survey[['sensor_id', common_col]], on='sensor_id', how='left').dropna()
    coeff, pval = spearmanr(df_merged['pct'], df_merged[common_col])
    if print_output:
      print(f"Spearman correlation between % {source} (prob > {prob_thresh}, dBA > {loudness_min_dba}) and survey {common_col}:\nr = {coeff:.3f}, p = {pval:.3f}")
    return float(coeff), float(pval)


INTERMITTENT_DB_OFFSET = 10
MIN_INTERVALS_PER_HOUR = 15 * 20
CLASS_DETECTION_THRESHOLD = 0.5
INTERMITTENT_CLASSES = ['vehicle', 'aircraft', 'human', 'bird', 'siren']

class_sums = ",\n        ".join(
    f"SUM(({source} >= {CLASS_DETECTION_THRESHOLD})::BIGINT) AS {source}_detections"
    for source in INTERMITTENT_CLASSES
)
intermittent_query = f"""
    WITH hourly AS (
        SELECT
            sensor_id,
            date_trunc('hour', time) AS hour,
            avg_spl_a,
            {', '.join(INTERMITTENT_CLASSES)},
            COUNT(*) OVER (
                PARTITION BY sensor_id, date_trunc('hour', time)
            ) AS intervals_in_hour,
            AVG(avg_spl_a) OVER (
                PARTITION BY sensor_id, date_trunc('hour', time)
            ) AS mean_spl_energy
        FROM sources
        WHERE avg_spl_a > 0 AND sensor_id = ?
    ),
    intermittent AS (
        SELECT *
        FROM hourly
        WHERE intervals_in_hour >= {MIN_INTERVALS_PER_HOUR}
          AND 10 * LOG10(avg_spl_a)
              > 10 * LOG10(mean_spl_energy) + {INTERMITTENT_DB_OFFSET}
    )
    SELECT
        sensor_id,
        COUNT(*)::BIGINT AS n_intermittent_intervals,
        {class_sums}
    FROM intermittent
    GROUP BY sensor_id
    ORDER BY sensor_id
"""
# Each window is partitioned by sensor and hour. Batching sensors preserves
# the calculation while bounding the window operator's memory use.
sensor_ids = db.sql('SELECT DISTINCT sensor_id FROM sources ORDER BY sensor_id').df().sensor_id
intermittent_counts = pd.concat(
    [db.execute(intermittent_query, [sensor_id]).df() for sensor_id in sensor_ids],
    ignore_index=True,
)


survey_sensors = df_survey[['sensor_id']].dropna().drop_duplicates()
intermittent_pr_by_sensor = survey_sensors.merge(
    intermittent_counts, on='sensor_id', how='left'
)
count_columns = [
    'n_intermittent_intervals',
    *[f'{source}_detections' for source in INTERMITTENT_CLASSES],
]
intermittent_pr_by_sensor[count_columns] = (
    intermittent_pr_by_sensor[count_columns].fillna(0).astype('int64')
)
for source in INTERMITTENT_CLASSES:
    intermittent_pr_by_sensor[f'{source}_pr'] = np.where(
        intermittent_pr_by_sensor['n_intermittent_intervals'] > 0,
        intermittent_pr_by_sensor[f'{source}_detections']
        / intermittent_pr_by_sensor['n_intermittent_intervals'],
        np.nan,
    )

existing_counts = db.sql("SELECT * FROM intermittent_event_sums_per_sensor").df()
count_check = intermittent_counts.merge(
    existing_counts[['sensor_id', *INTERMITTENT_CLASSES]], on='sensor_id', how='left'
)
for source in INTERMITTENT_CLASSES:
    assert count_check[f'{source}_detections'].eq(count_check[source]).all()

support_columns = [
    'sensor_id', 'n_intermittent_intervals',
    *[f'{source}_pr' for source in INTERMITTENT_CLASSES],
]
display(intermittent_pr_by_sensor[support_columns].style.format({
    **{f'{source}_pr': '{:.4f}' for source in INTERMITTENT_CLASSES},
}))
print(
    'Sensors with an undefined intermittent ratio:',
    int(intermittent_pr_by_sensor['n_intermittent_intervals'].eq(0).sum()),
)


def intermittent_pr_survey_spearman(source, survey_column, print_output=True):
    paired = intermittent_pr_by_sensor[
        ['sensor_id', 'n_intermittent_intervals', f'{source}_pr']
    ].merge(
        df_survey[['sensor_id', survey_column]], on='sensor_id', how='left'
    ).dropna(subset=[f'{source}_pr', survey_column])
    result = spearmanr(paired[f'{source}_pr'], paired[survey_column])
    if print_output:
        print(
            f"Spearman correlation between intermittent conditional presence of "
            f"'{source}' and survey {survey_column}:\n"
            f"r = {result.statistic:.3f}, p = {result.pvalue:.3f}"
        )
    return float(result.statistic), float(result.pvalue)



In [ ]:
mapping = {
  'vehicle': 'common_sources_traffic',
  'aircraft': 'common_sources_aviation',
  'human': 'common_sources_voices',
  'bird': 'common_sources_bird',
  'siren': 'common_sources_emergency',
}

rows = []
for col1, col2 in mapping.items():
  coeff_all, pval_all = source_survey_spearman(source=col1, common_col=col2, loudness_min_dba=0, print_output=False)
  coeff_60, pval_60 = source_survey_spearman(source=col1, common_col=col2, loudness_min_dba=60, print_output=False)
  coeff_intermittent, pval_intermittent = intermittent_pr_survey_spearman(source=col1, survey_column=col2, print_output=False)
  rows.append({
    'source': col1,
    'coeff_all': coeff_all,
    'pval_all': pval_all,
    'coeff_dba_gt_60': coeff_60,
    'pval_dba_gt_60': pval_60,
    'coeff_intermittent': coeff_intermittent,
    'pval_intermittent': pval_intermittent,
  })

spearman_pretty_table = pd.DataFrame(rows)
spearman_pretty_table = spearman_pretty_table[['source', 'coeff_all', 'coeff_dba_gt_60', 'coeff_intermittent']].rename(columns={
    'coeff_all': "Spearman's ρ",
    'coeff_dba_gt_60': "Spearman's ρ (dBA > 60)",
    'coeff_intermittent': "Spearman's ρ (intermittent events)",
})

correlation_columns = [
    "Spearman's ρ",
    "Spearman's ρ (dBA > 60)",
    "Spearman's ρ (intermittent events)",
]
spearman_pretty_table[correlation_columns] = spearman_pretty_table[correlation_columns].round(2)
spearman_pretty_table

## Noise annoyance distribution

In [ ]:
# Noise sensitivity score: sum items 1,3,4,5 + reverse-coded item 2 (6 - x)

df_survey.loc[:, ['annoying_sources_traffic', 'annoying_sources_aviation',
       'annoying_sources_construction', 'annoying_sources_voices',
       'annoying_sources_neighbour', 'annoying_sources_emergency',
       'annoying_sources_other']].hist(figsize=(12, 7))

In [ ]:
df_survey.loc[:, ['annoying_sources_traffic', 'annoying_sources_aviation',
       'annoying_sources_construction', 'annoying_sources_voices',
       'annoying_sources_neighbour', 'annoying_sources_emergency',
       'annoying_sources_other']].describe()

## Spearman correlation of annoyance and sensor detections

In [ ]:
mapping = {
  'vehicle': 'annoying_sources_traffic',
  'aircraft': 'annoying_sources_aviation',
  'human': 'annoying_sources_voices',
  'siren': 'annoying_sources_emergency',
}

rows = []
for col1, col2 in mapping.items():
  coeff_all, pval_all = source_survey_spearman(source=col1, common_col=col2, loudness_min_dba=0, print_output=False)
  coeff_60, pval_60 = source_survey_spearman(source=col1, common_col=col2, loudness_min_dba=60, print_output=False)
  coeff_intermittent, pval_intermittent = intermittent_pr_survey_spearman(source=col1, survey_column=col2, print_output=False)
  rows.append({
    'source': col1,
    'coeff_all': coeff_all,
    'pval_all': pval_all,
    'coeff_dba_gt_60': coeff_60,
    'pval_dba_gt_60': pval_60,
    'coeff_intermittent': coeff_intermittent,
    'pval_intermittent': pval_intermittent,
  })

spearman_pretty_table = pd.DataFrame(rows)
spearman_pretty_table = spearman_pretty_table[['source', 'coeff_all', 'coeff_dba_gt_60', 'coeff_intermittent']].rename(columns={
    'coeff_all': "Spearman's ρ",
    'coeff_dba_gt_60': "Spearman's ρ (dBA > 60)",
    'coeff_intermittent': "Spearman's ρ (intermittent events)",
})

correlation_columns = [
    "Spearman's ρ",
    "Spearman's ρ (dBA > 60)",
    "Spearman's ρ (intermittent events)",
]
spearman_pretty_table[correlation_columns] = spearman_pretty_table[correlation_columns].round(2)
spearman_pretty_table

In [ ]:
mapping = {
  'common_sources_traffic': 'annoying_sources_traffic',
  'common_sources_aviation': 'annoying_sources_aviation',
  'common_sources_voices': 'annoying_sources_voices',
  'common_sources_emergency': 'annoying_sources_emergency',
}

rows = []
for col1, col2 in mapping.items():
  coeff_all, pval_all =  spearmanr(df_survey[col1], df_survey[col2])
  rows.append({
    'source': col1,
    "Spearman's ρ": coeff_all,
    'pval': pval_all,
  })

spearman_pretty_table = pd.DataFrame(rows)

spearman_pretty_table["Spearman's ρ"] = spearman_pretty_table["Spearman's ρ"].round(2)

spearman_pretty_table = spearman_pretty_table[['source', "Spearman's ρ"]]
spearman_pretty_table

## Noise sensitivity and average loudness

In [ ]:
# average loudness per sensor
dba = db.sql("SELECT sensor_id, 10 * log10(AVG(avg_spla)) as avg_dba FROM loudness_per_minute GROUP BY sensor_id").df()

# merge noise sensitivity score with avg_dba
df = df_survey.merge(dba, on='sensor_id', how='left')[['noise_sensitivity_score', 'avg_dba']]

# Spearman correlation
coeff, pval = spearmanr(df['noise_sensitivity_score'], df['avg_dba'])
print(f"Spearman correlation between noise sensitivity score and average dBA")
print(f"coeff = {coeff:.2f}, pval = {pval:.3f}")

## Spearman correlation between IR and survey

In [ ]:
ir_per_hour = db.sql("SELECT * from ir_per_hour").df()
ir_per_sensor = ir_per_hour.groupby('sensor_id')['intermittency_ratio'].mean().reset_index().sort_values('sensor_id', ascending=False)

feature = 'annoying_features_bursts'
df_merged = ir_per_sensor.merge(df_survey[['sensor_id', feature]], on='sensor_id', how='left').dropna()

coeff, pval =  spearmanr(df_merged[feature], df_merged['intermittency_ratio'])
print(f"Spearman correlation between annoyance from short bursts and measured intermittency ratio")
print(f"coeff = {coeff:.2f}, pval = {pval:.3f}")

In [ ]:
df_merged = db.sql(f"""
  SELECT
    sensor_id,
    10 * LOG10(MEDIAN(median_spla)) as median_dba
  FROM loudness_per_hour
  GROUP BY sensor_id
""").df().merge(df_survey[['sensor_id', 'annoying_features_volume']], on='sensor_id', how='left').dropna()

coeff, pval =  spearmanr(df_merged['annoying_features_volume'], df_merged['median_dba'])
print(f"Spearman correlation between annoyance from volume and median dba")
print(f"coeff = {coeff:.2f}, pval = {pval:.3f}")

## Benjamini-Hochberg correction across all 33 survey correlations

The 15 correlations in Table 2, 16 correlations in Table 3, and two additional narrative correlations are treated as one family. Two-sided permutation p-values are calculated with 9,999 paired-label permutations and adjusted using Benjamini-Hochberg at an FDR of 0.05.

In [ ]:
N_PERMUTATION_RESAMPLES = 9_999
PERMUTATION_SEED = 1
FDR_ALPHA = 0.05


def _sensor_pair_values(source, survey_column, loudness_min_dba):
    paired = db.sql(f"""
        SELECT
          sensor_id,
          100.0 * AVG(({source} > 0.5)::DOUBLE) AS pct
        FROM sources
        WHERE avg_dba > {loudness_min_dba}
        GROUP BY sensor_id
    """).df().merge(
        df_survey[['sensor_id', survey_column]], on='sensor_id', how='left'
    ).dropna()
    paired = paired.sort_values('sensor_id', kind='stable')
    return paired['pct'].to_numpy(dtype=float), paired[survey_column].to_numpy(dtype=float)


def _intermittent_pair_values(source, survey_column):
    paired = intermittent_pr_by_sensor[
        ['sensor_id', 'n_intermittent_intervals', f'{source}_pr']
    ].merge(
        df_survey[['sensor_id', survey_column]], on='sensor_id', how='left'
    ).dropna(subset=[f'{source}_pr', survey_column])
    paired = paired.sort_values('sensor_id', kind='stable')
    return paired[f'{source}_pr'].to_numpy(dtype=float), paired[survey_column].to_numpy(dtype=float)


def _permutation_spearman(x, y, seed):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape or np.unique(x).size < 2 or np.unique(y).size < 2:
        raise ValueError('Spearman correlation requires paired, non-constant variables.')

    rho = float(spearmanr(x, y).statistic)
    x_rank = rankdata(x)
    y_rank = rankdata(y)
    y_centered = y_rank - y_rank.mean()
    y_sum_squares = np.sum(y_centered ** 2)

    def statistic(permuted_x_rank, axis=-1):
        x_centered = permuted_x_rank - np.mean(permuted_x_rank, axis=axis, keepdims=True)
        return np.sum(x_centered * y_centered, axis=axis) / np.sqrt(
            np.sum(x_centered ** 2, axis=axis) * y_sum_squares
        )

    result = permutation_test(
        (x_rank,),
        statistic,
        permutation_type='pairings',
        vectorized=True,
        n_resamples=N_PERMUTATION_RESAMPLES,
        batch=2_048,
        alternative='two-sided',
        rng=np.random.default_rng(seed),
    )
    if not np.isclose(result.statistic, rho, rtol=0, atol=1e-12):
        raise AssertionError('Permutation statistic does not reproduce Spearman rho.')
    return {'n': int(x.size), 'rho': rho, 'p_permutation': float(result.pvalue)}


records = []


def _add_result(table, source_label, indicator, x, y):
    result = _permutation_spearman(x, y, seed=PERMUTATION_SEED)
    records.append({
        'table': table,
        'source': source_label,
        'indicator': indicator,
        **result,
    })


table2_specs = [
    ('vehicle', 'Vehicles', 'common_sources_traffic'),
    ('aircraft', 'Aviation', 'common_sources_aviation'),
    ('human', 'Speech/crowd', 'common_sources_voices'),
    ('bird', 'Bird song', 'common_sources_bird'),
    ('siren', 'Sirens', 'common_sources_emergency'),
]
for source, source_label, survey_column in table2_specs:
    x, y = _sensor_pair_values(source, survey_column, loudness_min_dba=0)
    _add_result('Table 2', source_label, 'Presence ratio', x, y)
    x, y = _sensor_pair_values(source, survey_column, loudness_min_dba=60)
    _add_result('Table 2', source_label, 'Presence ratio >60 dB(A)', x, y)
    x, y = _intermittent_pair_values(source, survey_column)
    _add_result('Table 2', source_label, 'Conditional presence ratio in intermittent intervals', x, y)

table3_specs = [
    ('vehicle', 'Vehicles', 'common_sources_traffic', 'annoying_sources_traffic'),
    ('aircraft', 'Aviation', 'common_sources_aviation', 'annoying_sources_aviation'),
    ('human', 'Speech/crowd', 'common_sources_voices', 'annoying_sources_voices'),
    ('siren', 'Sirens', 'common_sources_emergency', 'annoying_sources_emergency'),
]
for source, source_label, perceived_column, annoyance_column in table3_specs:
    paired = df_survey[['sensor_id', perceived_column, annoyance_column]].dropna().sort_values('sensor_id', kind='stable')
    _add_result(
        'Table 3',
        source_label,
        'Perceived source prevalence',
        paired[perceived_column].to_numpy(dtype=float),
        paired[annoyance_column].to_numpy(dtype=float),
    )
    x, y = _sensor_pair_values(source, annoyance_column, loudness_min_dba=0)
    _add_result('Table 3', source_label, 'Measured presence ratio', x, y)
    x, y = _sensor_pair_values(source, annoyance_column, loudness_min_dba=60)
    _add_result('Table 3', source_label, 'Measured presence ratio >60 dB(A)', x, y)
    x, y = _intermittent_pair_values(source, annoyance_column)
    _add_result('Table 3', source_label, 'Measured conditional presence ratio in intermittent intervals', x, y)

intermittency_pairs = db.sql(
    "SELECT sensor_id, AVG(intermittency_ratio) AS intermittency_ratio "
    "FROM ir_per_hour GROUP BY sensor_id"
).df().merge(
    df_survey[['sensor_id', 'annoying_features_bursts']], on='sensor_id', how='left'
).dropna().sort_values('sensor_id', kind='stable')
_add_result(
    'Narrative',
    'Overall intermittency',
    'Intermittency ratio vs short-burst annoyance',
    intermittency_pairs['intermittency_ratio'].to_numpy(dtype=float),
    intermittency_pairs['annoying_features_bursts'].to_numpy(dtype=float),
)

volume_pairs = db.sql("""
    SELECT
      sensor_id,
      10 * LOG10(MEDIAN(median_spla)) AS median_dba
    FROM loudness_per_hour
    GROUP BY sensor_id
""").df().merge(
    df_survey[['sensor_id', 'annoying_features_volume']], on='sensor_id', how='left'
).dropna().sort_values('sensor_id', kind='stable')
_add_result(
    'Narrative',
    'Median loudness',
    'Median dB(A) vs perceived-volume annoyance',
    volume_pairs['median_dba'].to_numpy(dtype=float),
    volume_pairs['annoying_features_volume'].to_numpy(dtype=float),
)

bh_33_results = pd.DataFrame(records)
assert bh_33_results['table'].value_counts().to_dict() == {
    'Table 3': 16, 'Table 2': 15, 'Narrative': 2
}
assert bh_33_results['n'].eq(n_unique_sensors).all()

reject_bh, p_bh, _, _ = multipletests(
    bh_33_results['p_permutation'], alpha=FDR_ALPHA, method='fdr_bh'
)
bh_33_results['p_bh'] = p_bh
bh_33_results['significant_bh'] = reject_bh
assert bh_33_results['p_bh'].between(0, 1).all()

display(bh_33_results.style.format({
    'rho': '{:.3f}',
    'p_permutation': '{:.6f}',
    'p_bh': '{:.6f}',
}))
print(f"BH-significant correlations: {int(bh_33_results['significant_bh'].sum())} of 33")